# Fiber Photometry Behavior Analysis
Runs the same analyses as `run_open_field_analysis.py`, but only after filtering the raw TSV files to sessions whose task name contains neither `mag` nor `pretraining` (case-insensitive). Sessions without a task name are skipped because they cannot pass the check.

In [1]:
import importlib
import sys
from pathlib import Path
from unittest.mock import patch

repo_root = Path.cwd()
if not (repo_root / "src").is_dir():
    repo_root = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(repo_root))

import_data_module = importlib.import_module("src.behavior_import.import_data")

task = "open-field"
cohort_id = 4
cohort = "cohort-0{cohort_id}".format(cohort_id=cohort_id)
folder_name = "3x3_field_magnitude_bandit"
root = f"/Volumes/behrens/meg/{folder_name}/rawdata/{cohort}/"
excluded_task_name_parts = ("mag", "pretraining")
problem_numbers = None # Use None to analyze every available problem.

# Analysis-family toggles
run_choice_probability_analysis = False
run_first_leave_analysis = True
run_number_of_reversals_analysis = True
run_rank_proportion_analysis = True
run_single_session_analysis = True

# Choice-probability sub-analysis toggles
choice_run_all = True
choice_skip_initial_trials = False
choice_moving_average = True
choice_remove_bad = True
choice_split_by_best_change = False
choice_split_by_best_change_and_half = False
choice_split_by_first_two = False
choice_split_by_half = False


In [2]:
def task_name_from_df(df):
    required_columns = {"type", "subtype", "content"}
    if not required_columns.issubset(df.columns):
        return None

    task_name_rows = df.loc[
        df["type"].eq("info") & df["subtype"].eq("task_name"),
        "content",
    ]
    if task_name_rows.empty:
        return None
    return str(task_name_rows.iloc[0])


all_tsv_files = import_data_module.collect_tsv_files(Path(root).resolve())
eligible_tsv_files = []
for tsv_path, df in all_tsv_files:
    task_name = task_name_from_df(df)
    if task_name is None:
        print(f"[WARN] Skipping {tsv_path}: no task name found.")
        continue
    if any(part in task_name.casefold() for part in excluded_task_name_parts):
        print(f"[INFO] Skipping {tsv_path}: excluded task name {task_name!r}.")
        continue
    eligible_tsv_files.append((tsv_path, df))

print(f"[INFO] Keeping {len(eligible_tsv_files)} of {len(all_tsv_files)} TSV file(s).")
with patch.object(import_data_module, "collect_tsv_files", return_value=eligible_tsv_files):
    subjects_data = import_data_module.import_data(root)

if not subjects_data:
    raise ValueError("No sessions remain after filtering task names.")


[INFO] Keeping 110 of 110 TSV file(s).
[INFO] Processed 13 subjects(s), 104 session(s).


In [3]:
# Reuse the analysis body so this notebook stays in sync with the open-field pipeline.
pipeline_path = repo_root / "scripts" / "full_pipelines" / "run_open_field_analysis.py"
pipeline_source = pipeline_path.read_text()
imports_source = pipeline_source[:pipeline_source.index('task = "open-field"')]
exec(compile(imports_source, str(pipeline_path), "exec"), globals())
analysis_marker = "# Choice Probability Analysis"
analysis_source = pipeline_source[pipeline_source.index(analysis_marker):]
original_data_setup = (
    'root = f"/Volumes/behrens/meg/{folder_name}/{cohort}/rawdata/"\n'
    'subjects_data = import_data(root)\n'
    'subjects_trials_by_problem = extract_trials_grouped_by_problem(subjects_data)\n'
)
filtered_data_setup = (
    'subjects_trials_by_problem = extract_trials_grouped_by_problem(subjects_data)\n'
    + 'if problem_numbers is not None:\n'
    + '    requested_problem_numbers = set(problem_numbers)\n'
    + '    missing_problem_numbers = requested_problem_numbers - set(subjects_trials_by_problem)\n'
    + '    if missing_problem_numbers:\n'
    + '        raise ValueError(f"Requested problem number(s) not found: {sorted(missing_problem_numbers)}")\n'
    + '    subjects_trials_by_problem = {number: trials for number, trials in subjects_trials_by_problem.items() if number in requested_problem_numbers}\n'
    + '    print(f"[INFO] Analyzing problem number(s): {sorted(subjects_trials_by_problem)}")\n'
)
if original_data_setup not in analysis_source:
    raise RuntimeError("The open-field pipeline data setup has changed.")
analysis_source = filtered_data_setup + "\n" + analysis_source.replace(original_data_setup, "", 1)

choice_toggle_replacements = {
    "run_all = True": "run_all = choice_run_all",
    "skip = True": "skip = choice_skip_initial_trials",
    "moving_avg = True": "moving_avg = choice_moving_average",
    "remove_bad = True": "remove_bad = choice_remove_bad",
    "split_by_best_change = False": "split_by_best_change = choice_split_by_best_change",
    "split_by_best_change_and_half = False": "split_by_best_change_and_half = choice_split_by_best_change_and_half",
    "split_by_first_two = True": "split_by_first_two = choice_split_by_first_two",
    "split_by_half = True": "split_by_half = choice_split_by_half",
}
for pipeline_setting, notebook_setting in choice_toggle_replacements.items():
    if pipeline_setting not in analysis_source:
        raise RuntimeError(f"The open-field pipeline setting has changed: {pipeline_setting}")
    analysis_source = analysis_source.replace(pipeline_setting, notebook_setting, 1)

def toggle_section(source, marker, next_marker, enabled, label):
    if enabled:
        return source
    start = source.index(marker)
    end = source.index(next_marker, start) if next_marker else len(source)
    replacement = f'{marker}\n\nprint("[INFO] Skipping {label}.")\n\n'
    return source[:start] + replacement + source[end:]

analysis_sections = [
    ("# Choice Probability Analysis", "# First Leave Analysis", run_choice_probability_analysis, "choice probability analysis"),
    ("# First Leave Analysis", "# Number of Reversals Analysis", run_first_leave_analysis, "first leave analysis"),
    ("# Number of Reversals Analysis", "# Rank Proportion Analysis", run_number_of_reversals_analysis, "number of reversals analysis"),
    ("# Rank Proportion Analysis", "# Single Session Analysis", run_rank_proportion_analysis, "rank proportion analysis"),
    ("# Single Session Analysis", None, run_single_session_analysis, "single session analysis"),
]
for section in reversed(analysis_sections):
    analysis_source = toggle_section(analysis_source, *section)
output_root = repo_root / "results" / "figures"
analysis_source = analysis_source.replace("../../results/figures", str(output_root))
print(f"[INFO] Saving analysis files under {output_root / task / cohort}")
exec(compile(analysis_source, str(pipeline_path), "exec"), globals())


[INFO] Saving analysis files under /Users/megyoung/magnitude-bandit-analysis/results/figures/open-field/cohort-04
[WARNING] No trial information found for subject RT_176_N, session ses-1_date-20260908
[WARNING] No trial information found for subject RT_176_N, session ses-2_date-20260908
[WARNING] No trial information found for subject RT_176_N, session ses-3_date-20260909
[WARNING] No trial information found for subject RT_176_N, session ses-4_date-20260909
[WARNING] No trial information found for subject RT_176_N, session ses-5_date-20260910
[WARNING] No trial information found for subject RT_176_N, session ses-6_date-20260910
[WARNING] No trial information found for subject RT_176_L, session ses-1_date-20260908
[WARNING] No trial information found for subject RT_176_L, session ses-2_date-20260908
[WARNING] No trial information found for subject RT_176_L, session ses-4_date-20260909
[WARNING] No trial information found for subject RT_64_LR, session ses-1_date-20260908
[WARNING] No tri

/Users/megyoung/anaconda3/envs/magnitude-bandit-analysis/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/megyoung/anaconda3/envs/magnitude-bandit-analysis/lib/python3.13/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Running number of reversals analysis
1
RT_176_L: {'total_reversals': 0, 'good_reversals': 0, 'bad_reversals': 0}
RT_176_LR: {'total_reversals': 0, 'good_reversals': 0, 'bad_reversals': 0}
RT_176_N: {'total_reversals': 0, 'good_reversals': 0, 'bad_reversals': 0}
RT_176_R: {'total_reversals': 1, 'good_reversals': 0, 'bad_reversals': 1}
RT_177_L: {'total_reversals': 1, 'good_reversals': 1, 'bad_reversals': 0}
RT_177_LR: {'total_reversals': 0, 'good_reversals': 0, 'bad_reversals': 0}
RT_177_N: {'total_reversals': 1, 'good_reversals': 1, 'bad_reversals': 0}
RT_177_R: {'total_reversals': 0, 'good_reversals': 0, 'bad_reversals': 0}
RT_64_LR: {'total_reversals': 0, 'good_reversals': 0, 'bad_reversals': 0}
RT_64_N: {'total_reversals': 2, 'good_reversals': 2, 'bad_reversals': 0}
RT_64_R: {'total_reversals': 3, 'good_reversals': 2, 'bad_reversals': 1}
RT_65_LR: {'total_reversals': 2, 'good_reversals': 2, 'bad_reversals': 0}
RT_65_N: {'total_reversals': 3, 'good_reversals': 2, 'bad_reversals': 1}
